# Step 3 (Baseline YOLO) — Colab GPU Training

Notebook chạy tự động toàn diện (**Runtime ▸ Run all**) để huấn luyện B01 YOLO11n và B02 YOLO11s 100 epoch trên GPU Colab, trích xuất dự đoán calibration/fusion, và tải trọn gói artifact về máy local.

**Quy trình chuẩn bị trên Google Drive của bạn:**
1. Nén thư mục repo thành `PCB_Lab_impl3.zip` và upload lên Google Drive (`My Drive/`).
2. Nén thư mục dataset thành `DatasetVer4_Public.zip` (hoặc `DatasetVer4.zip`) và upload lên Google Drive (`My Drive/`).
3. Trên Colab: chọn **Runtime ▸ Change runtime type ▸ T4 GPU** (hoặc V100/A100).
4. Bấm **Runtime ▸ Run all** (hoặc chạy tuần tự từng cell).

> **Tối ưu tốc độ:** Notebook sẽ tự động tìm các file ZIP trên Drive, copy sang ổ đĩa SSD local `/content` và giải nén siêu tốc. Giúp tránh nghẽn I/O của Google Drive và tăng tốc huấn luyện gấp nhiều lần.

In [ ]:
# ===== Cell 1: Gate 0 — Hardware Verification =====
import os, sys, subprocess

def banner(t):
    print('=' * 60)
    print(t)
    print('=' * 60)

banner('Gate 0: CUDA hardware check')
!nvidia-smi || echo 'nvidia-smi not found'

import torch
cuda_ok = torch.cuda.is_available()
print('torch.version:', torch.__version__)
print('torch.cuda.is_available():', cuda_ok)
if cuda_ok:
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print('GPU[0]:', name, '(compute capability', cap, ')')
    cc = torch.cuda.get_device_capability(0)
    # Colab CUDA 12.x reports capability >= 7.0 for T4/V100/A100
    assert cc[0] >= 7, f'GPU compute capability {cc} too low for AMP/torch CUDA'
    print('CUDA compute capability OK for training')
else:
    raise SystemExit('Gate 0 FAILED: no CUDA GPU. Change runtime to GPU.')

In [ ]:
# ===== Cell 2: Dataset Setup (Auto-extract from Drive ZIP) =====
import os, shutil, zipfile, glob
from pathlib import Path
from google.colab import drive

# 1. Kết nối Google Drive
drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive')

# 2. Tìm tệp zip dataset trên Drive
possible_zips = (
    list(drive_root.glob('*[Dd]ataset*[Vv]er4*.zip')) +
    list(drive_root.glob('*[Dd]ataset*.zip')) +
    list(drive_root.glob('*[Dd]eep[Pp][Cc][Bb]*.zip'))
)

local_ds_dir = Path('/content/DatasetVer4_Public')

if possible_zips:
    ds_zip = possible_zips[0]
    print(f'--> Tìm thấy tệp ZIP dataset: {ds_zip.name}')
    local_zip = Path(f'/content/{ds_zip.name}')
    if not local_zip.exists():
        print('--> Đang sao chép dataset ZIP sang SSD local (/content)...')
        shutil.copy2(ds_zip, local_zip)
    
    temp_extract = Path('/content/ds_temp')
    if temp_extract.exists():
        shutil.rmtree(temp_extract)
    
    print('--> Đang giải nén dataset tại local...')
    with zipfile.ZipFile(local_zip, 'r') as zf:
        zf.extractall(temp_extract)
    
    # Tìm chính xác vị trí chứa manifest samples.jsonl
    manifest_files = list(temp_extract.rglob('samples.jsonl'))
    if manifest_files:
        # benchmarks/deeppcb/manifests/samples.jsonl -> root là parent thứ 4
        extracted_root = manifest_files[0].parents[3]
        if local_ds_dir.exists():
            shutil.rmtree(local_ds_dir)
        shutil.move(str(extracted_root), str(local_ds_dir))
        shutil.rmtree(temp_extract, ignore_errors=True)
        if local_zip.exists():
            os.remove(local_zip)
        print(f'--> Giải nén dataset thành công vào: {local_ds_dir}')
    else:
        raise FileNotFoundError('Không tìm thấy samples.jsonl bên trong file ZIP dataset!')
else:
    # Dự phòng nếu người dùng upload thư mục giải nén sẵn
    possible_dirs = list(drive_root.glob('*[Dd]ataset*[Vv]er4*'))
    if possible_dirs and possible_dirs[0].is_dir():
        local_ds_dir = possible_dirs[0]
        print(f'--> Sử dụng trực tiếp thư mục dataset trên Drive: {local_ds_dir}')
    else:
        raise FileNotFoundError('Không tìm thấy file ZIP hoặc thư mục DatasetVer4 trên Google Drive!')

DATASET_ROOT = str(local_ds_dir)
os.environ['DATASET_ROOT'] = DATASET_ROOT
manifest = Path(DATASET_ROOT) / 'benchmarks' / 'deeppcb' / 'manifests' / 'samples.jsonl'
assert manifest.exists(), f'Lỗi: Không tìm thấy manifest tại {manifest}'
print(f'==> DATASET_ROOT hợp lệ: {DATASET_ROOT}')
print(f'==> Manifest OK ({manifest.stat().st_size} bytes)')


In [ ]:
# ===== Cell 3: Codebase Setup & Environment Install =====
import os, shutil, zipfile, sys, re
from pathlib import Path

drive_root = Path('/content/drive/MyDrive')
local_repo = Path('/content/PCB_Lab_impl3')

# 1. Dọn dẹp nếu có thư mục cũ hoặc symlink hỏng
if local_repo.exists() or local_repo.is_symlink():
    if local_repo.is_symlink():
        os.unlink(local_repo)
    else:
        shutil.rmtree(local_repo, ignore_errors=True)

# 2. Tìm tệp zip của repo trên Drive
repo_zips = (
    list(drive_root.glob('*PCB_Lab_impl3*.zip')) +
    list(drive_root.glob('*colab_bundle*.zip')) +
    list(drive_root.glob('*step3*.zip'))
)

if repo_zips:
    chosen_zip = repo_zips[0]
    print(f'--> Tìm thấy tệp ZIP mã nguồn: {chosen_zip.name}')
    local_zip_temp = Path('/content/repo_temp.zip')
    shutil.copy2(chosen_zip, local_zip_temp)
    
    temp_dir = Path('/content/repo_extracted_temp')
    if temp_dir.exists():
        shutil.rmtree(temp_dir)
    with zipfile.ZipFile(local_zip_temp, 'r') as zf:
        zf.extractall(temp_dir)
    
    # Kiểm tra xem zip có chứa folder lồng (ví dụ PCB_Lab_impl3/...) hay không
    subdirs = [d for d in temp_dir.iterdir() if d.is_dir() and (d / 'pyproject.toml').exists()]
    if subdirs:
        shutil.move(str(subdirs[0]), str(local_repo))
        shutil.rmtree(temp_dir, ignore_errors=True)
    elif (temp_dir / 'pyproject.toml').exists():
        shutil.move(str(temp_dir), str(local_repo))
    else:
        # Tìm bất kỳ thư mục nào chứa pyproject.toml
        pyprojects = list(temp_dir.rglob('pyproject.toml'))
        if pyprojects:
            shutil.move(str(pyprojects[0].parent), str(local_repo))
            shutil.rmtree(temp_dir, ignore_errors=True)
        else:
            raise FileNotFoundError('Không tìm thấy pyproject.toml trong file ZIP repo!')
    
    if local_zip_temp.exists():
        os.remove(local_zip_temp)
    print(f'--> Đã giải nén repo vào: {local_repo}')
else:
    # Dự phòng nếu dùng thư mục thô trên Drive
    drive_dirs = list(drive_root.glob('*PCB_Lab_impl3*'))
    if drive_dirs and drive_dirs[0].is_dir():
        print(f'--> Tạo symlink tới thư mục repo trên Drive: {drive_dirs[0]}')
        os.symlink(drive_dirs[0], local_repo)
    else:
        raise FileNotFoundError('Không tìm thấy file ZIP PCB_Lab_impl3.zip hoặc thư mục PCB_Lab_impl3 trên Drive!')

# 3. Thiết lập thư mục làm việc và nạp src-layout vào sys.path
os.chdir(str(local_repo))
src_dir = str(local_repo / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
if str(local_repo) not in sys.path:
    sys.path.insert(0, str(local_repo))

# Đặt PYTHONPATH để các subprocess (train, extract) luôn nhận diện module
os.environ['PYTHONPATH'] = f"{src_dir}:{str(local_repo)}:{os.environ.get('PYTHONPATH', '')}"
print('Thư mục làm việc hiện tại:', os.getcwd())
print('Đã nạp đường dẫn src vào sys.path:', src_dir)

# 4. Cài đặt các thư viện (bảo toàn PyTorch CUDA của Colab)
req_path = local_repo / 'requirements' / 'step3.txt'
assert req_path.exists(), f'Lỗi: Không tìm thấy {req_path}'

skip = {'torch', 'torchvision'}
kept = []
for line in open(req_path, encoding='utf-8'):
    s = line.strip()
    if not s or s.startswith('#'):
        continue
    pkg = re.split(r'[=<>!~ ]', s)[0].strip().lower()
    if pkg in skip:
        print('SKIP (sử dụng PyTorch CUDA có sẵn của Colab):', s)
        continue
    kept.append(s)

filtered = '/content/step3_filtered.txt'
open(filtered, 'w', encoding='utf-8').write('\n'.join(kept) + '\n')

print('Đang cài đặt các thư viện bổ sung...')
!pip install -q -r {filtered}
!pip install -q --no-build-isolation --no-deps -e .

import pcb_lab
print('--> [THÀNH CÔNG] Đã import pcb_lab! Phiên bản:', getattr(pcb_lab, '__version__', '0.1.0'))


In [ ]:
# ===== Cell 4: Download Pretrained Weights =====
from pathlib import Path
import urllib.request

PRETRAINED_DIR = Path('artifacts/pretrained')
PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)

ASSET_BASE = 'https://github.com/ultralytics/assets/releases/download/v8.3.0'
files = ['yolo11n.pt', 'yolo11s.pt']

for fn in files:
    dst = PRETRAINED_DIR / fn
    if dst.exists() and dst.stat().st_size > 1_000_000:
        print('already present:', dst)
        continue
    url = f'{ASSET_BASE}/{fn}'
    print('downloading', url)
    urllib.request.urlretrieve(url, dst)
    print('  ->', dst, dst.stat().st_size, 'bytes')

print('Pretrained ready:', sorted(p.name for p in PRETRAINED_DIR.glob('*.pt')))

In [ ]:
# ===== Cell 5: Train B01 YOLO11n (100 epochs, seed 42, AdamW, AMP auto-on GPU) =====
# --allow-download: fallback fetch if Cell 4 weights are missing.
# AMP: omitted -> auto-enabled on CUDA GPU (off on CPU). --device 0 = first GPU.
!python scripts/train_yolo.py --config configs/models/yolo11n.yaml --seed 42 --device 0 --allow-download

from pathlib import Path
print('B01 artifact:')
for p in sorted(Path('artifacts/yolo/B01_yolo11n/seed42').iterdir()):
    print('  ', p.name, p.stat().st_size)

In [ ]:
# ===== Cell 6: Train B02 YOLO11s (100 epochs, seed 42, AdamW, AMP auto-on GPU) =====
!python scripts/train_yolo.py --config configs/models/yolo11s.yaml --seed 42 --device 0 --allow-download

from pathlib import Path
print('B02 artifact:')
for p in sorted(Path('artifacts/yolo/B02_yolo11s/seed42').iterdir()):
    print('  ', p.name, p.stat().st_size)

In [ ]:
# ===== Cell 7: Extract Predictions (calibration + fusion only; test is blocked) =====
print('--- B01 YOLO11n ---')
!python scripts/extract_yolo_predictions.py --artifact artifacts/yolo/B01_yolo11n/seed42 --partitions calibration fusion
print('--- B02 YOLO11s ---')
!python scripts/extract_yolo_predictions.py --artifact artifacts/yolo/B02_yolo11s/seed42 --partitions calibration fusion

from pathlib import Path
print('\nExtracted prediction files:')
for p in sorted(Path('artifacts/yolo').rglob('preds_*.jsonl')):
    print('  ', p, p.stat().st_size, 'bytes')

In [ ]:
# ===== Cell 8: Export Bundle (download artifacts back to local) =====
import shutil, os
from pathlib import Path
from google.colab import files

bundle = 'yolo_step3_artifacts.zip'
if Path(bundle).exists():
    os.remove(bundle)
shutil.make_archive('yolo_step3_artifacts', 'zip', 'artifacts/yolo')
print('Created', bundle, Path(bundle).stat().st_size, 'bytes')
print('Contents (top-level model dirs):')
for p in sorted(Path('artifacts/yolo').iterdir()):
    print('  ', p.name)
files.download(bundle)
print('Download triggered. Save yolo_step3_artifacts.zip to your local machine.')